In [1]:
import pandas as pd 

df_sales = pd.read_csv('../data/processed/online_retail_cleaned.csv')
df_cancellations = pd.read_csv('../data/processed/cancellations.csv')

all_transactions = pd.concat([df_sales, df_cancellations], ignore_index=True)
all_transactions['InvoiceDate'] = pd.to_datetime(all_transactions['InvoiceDate'])

print(all_transactions.shape)

(1022941, 11)


In [2]:
customers = all_transactions.dropna(subset=['Customer ID']).groupby('Customer ID').agg(
    country=('Country', 'first')
).reset_index()

customers.columns = ['customer_id', 'country']
customers['customer_id'] = customers['customer_id'].astype(int)

customers.to_csv('../data/processed/customers.csv', index=False)
print(customers.shape)
customers.head()                                    

(5928, 2)


,customer_id,country
0,12346,United Kingdom
1,12347,Iceland
2,12348,Finland
3,12349,Italy
4,12350,Norway


In [3]:
products = all_transactions.groupby('StockCode').agg(
    description=('Description', 'first'),
    category=('Category', 'first')
).reset_index()

products.columns = ['stock_code', 'description', 'category']

products.to_csv('../data/processed/products.csv', index=False)
print(products.shape)
products.head()

(4759, 3)


,stock_code,description,category
0,10002,INFLATABLE POLITICAL GLOBE,Toys & Kids Craft
1,10002R,ROBOT PENCIL SHARPNER,Stationery & Gift Wrap
2,10080,GROOVY CACTUS INFLATABLE,Toys & Kids Craft
3,10109,BENDY COLOUR PENCILS,Stationery & Gift Wrap
4,10120,DOGGY RUBBER,Other


In [4]:
invoices = all_transactions.groupby('Invoice').agg(
    customer_id=('Customer ID', 'first'),
    invoice_date=('InvoiceDate', 'min'),
    is_cancelled=('is_cancelled', 'first')
).reset_index()

invoices.columns = ['invoice_id', 'customer_id', 'invoice_date', 'is_cancelled']

#convert True/False to 1/0 so MySQL's BOOLEAN column accepts.
invoices['is_cancelled'] = invoices['is_cancelled'].astype(int)

#use nullable integer type so missing cusotmer_ids stay as true empty values 
invoices['customer_id'] = invoices['customer_id'].astype('Int64')

invoices.to_csv('../data/processed/invoices.csv', index=False)
print(invoices.shape)
invoices.head()

(47821, 4)


,invoice_id,customer_id,invoice_date,is_cancelled
0,489434,13085,2009-12-01 07:45:00,0
1,489435,13085,2009-12-01 07:46:00,0
2,489436,13078,2009-12-01 09:06:00,0
3,489437,15362,2009-12-01 09:08:00,0
4,489438,18102,2009-12-01 09:24:00,0


In [5]:
invoice_items = all_transactions[['Invoice', 'StockCode', 'Quantity', 'Price', 'Revenue']].reset_index()
invoice_items.columns = ['invoice_item_id', 'invoice_id', 'stock_code', 'quantity', 'price', 'revenue']

invoice_items.to_csv('../data/processed/invoice_items.csv', index=False)
print(invoice_items.shape)
invoice_items.head()

(1022941, 6)


,invoice_item_id,invoice_id,stock_code,quantity,price,revenue
0,0,489434,85048,12,6.95,83.4
1,1,489434,79323P,12,6.75,81.0
2,2,489434,79323W,12,6.75,81.0
3,3,489434,22041,48,2.10,100.8
4,4,489434,21232,24,1.25,30.0


In [6]:
invoices.head()

,invoice_id,customer_id,invoice_date,is_cancelled
0,489434,13085,2009-12-01 07:45:00,0
1,489435,13085,2009-12-01 07:46:00,0
2,489436,13078,2009-12-01 09:06:00,0
3,489437,15362,2009-12-01 09:08:00,0
4,489438,18102,2009-12-01 09:24:00,0


In [7]:
invoices = pd.read_csv('../data/processed/invoices.csv')

with open('../sql/insert_invoices.sql', 'w') as f:
    f.write("USE nolan_retail_co;\n\n")
    for _, row in invoices.iterrows():
        invoice_id = row['invoice_id']
        # Explicitly write the SQL keyword NULL when customer_id is missing
        if pd.isna(row['customer_id']):
            customer_id = 'NULL'
        else:
            customer_id = int(row['customer_id'])
        invoice_date = row['invoice_date']
        is_cancelled = int(row['is_cancelled'])
        f.write(f"INSERT INTO invoices (invoice_id, customer_id, invoice_date, is_cancelled) "
                f"VALUES ('{invoice_id}', {customer_id}, '{invoice_date}', {is_cancelled});\n")

print("SQL file generated:", len(invoices), "rows")

SQL file generated: 47821 rows


In [8]:
with open('../sql/insert_invoice_items.sql', 'w') as f:
    f.write("USE nolan_retail_co;\n\n")
    for _, row in invoice_items.iterrows():
        invoice_id = row['invoice_id']
        stock_code = row['stock_code']
        quantity = int(row['quantity'])
        price = row['price']
        revenue = row['revenue']
        f.write(f"INSERT INTO invoice_items (invoice_id, stock_code, quantity, price, revenue) "
                f"VALUES ('{invoice_id}', '{stock_code}', {quantity}, {price}, {revenue});\n")

print("SQL file generated:", len(invoice_items), "rows")

SQL file generated: 1022941 rows


In [9]:
missing_stock_codes = set(invoice_items['stock_code']) - set(products['stock_code'])
print(f"{len(missing_stock_codes)} stock codes in invoice_items are missing from products")
print(list(missing_stock_codes)[:20])

0 stock codes in invoice_items are missing from products
[]


In [10]:
products = pd.read_csv('../data/processed/products.csv')

with open('../sql/insert_products.sql', 'w') as f:
    f.write("USE nolan_retail_co;\n\n")
    for _, row in products.iterrows():
        stock_code = row['stock_code']
        description = str(row['description']).replace("'", "''")
        category = row['category']
        f.write(f"INSERT INTO products (stock_code, description, category) "
                f"VALUES ('{stock_code}', '{description}', '{category}');\n")

print("SQL file generated:", len(products), "rows")

SQL file generated: 4759 rows


In [11]:
check_products = pd.read_csv('../data/processed/products.csv')
print(check_products.shape)
check_products.head()

(4759, 3)


,stock_code,description,category
0,10002,INFLATABLE POLITICAL GLOBE,Toys & Kids Craft
1,10002R,ROBOT PENCIL SHARPNER,Stationery & Gift Wrap
2,10080,GROOVY CACTUS INFLATABLE,Toys & Kids Craft
3,10109,BENDY COLOUR PENCILS,Stationery & Gift Wrap
4,10120,DOGGY RUBBER,Other


In [12]:
# Check for StockCodes that are identical except for letter case
products['stock_code_lower'] = products['stock_code'].str.lower()
case_duplicates = products[products.duplicated(subset='stock_code_lower', keep=False)]
case_duplicates.sort_values('stock_code_lower')

,stock_code,description,category,stock_code_lower


In [13]:
products = pd.read_csv('../data/processed/products.csv')
products['stock_code_lower'] = products['stock_code'].str.lower()
case_duplicates = products[products.duplicated(subset='stock_code_lower', keep=False)]
print(case_duplicates.shape)

(0, 4)


In [14]:
customers = pd.read_csv('../data/processed/customers.csv')

with open('../sql/insert_customers.sql', 'w') as f:
    f.write("USE nolan_retail_co;\n\n")
    for _, row in customers.iterrows():
        customer_id = int(row['customer_id'])
        country = str(row['country']).replace("'", "''")
        f.write(f"INSERT INTO customers (customer_id, country) "
                f"VALUES ({customer_id}, '{country}');\n")

print("SQL file generated:", len(customers), "rows")

SQL file generated: 5928 rows


In [15]:
products = pd.read_csv('../data/processed/products.csv')
print(products.shape)
print(products['stock_code'].nunique())
print(products.duplicated(subset='stock_code').sum())

(4759, 3)
4759
0


In [16]:
print(invoice_items.shape)

(1022941, 6)


In [17]:
print(f"df_sales: {df_sales.shape[0]}")
print(f"df_cancellations: {df_cancellations.shape[0]}")
print(f"combined: {df_sales.shape[0] + df_cancellations.shape[0]}")

df_sales: 1003447
df_cancellations: 19494
combined: 1022941
